# VAE-cGAN — NBD — Subject-Dependent Split (scEEG -> iEEG)

**Split:** for EVERY subject individually, that subject's own segments are split
**70% train / 10% validation / 20% test** (a single split per subject, stratified by
IED label when the dataset has one). A **separate** encoder+generator+discriminator is
trained per subject. This tests how well scEEG->iEEG mapping works when the model has
already seen some of that same subject's own data — the easiest, most optimistic
evaluation setting.

*(Note: your original VAE-cGAN subject-dependent notebook used 5-fold CV per subject.
This rebuild uses a single 70/10/20 split per subject instead, to literally match what
you described ("70% training, 10% validation, 20% testing") and to match the
CNN-BiLSTM sibling notebook's split — simpler and 5x cheaper to run, at the cost of
each subject's test metric coming from one split instead of an average of 5.)*


### What changed in this "-nbd-28" rebuild (per your mam's feedback)

1. **New dataset.** Loads `segmented_dataset.npz` (was `balanced_segmented_dataset.npz`).
   The loader below looks up array names through a short alias list and prints exactly
   which keys it found, instead of silently assuming the old schema.
2. **Labels are now optional.** Your new dataset may not carry an IED/non-IED label at
   all. The loader detects this (`HAS_LABELS`) and automatically: (a) switches every
   stratified split to a plain/group split instead, and (b) drops the IED/Non-IED rows
   from the results table, reporting "Combined" only. Nothing below will crash if the
   label array is simply missing.
3. **Data-leakage fix.** Not applicable to this notebook — every split here stays **within one subject**, so there was never a cross-subject leakage risk here. The leakage bug your mam flagged was in the *subject-independent* notebook (`vae-cgan-nbd-pooledsplit-28`), which has the actual fix.
4. **[Round 2] Training cosine loss re-aligned to the exact reported COSSIM.** This
   was the single biggest issue flagged: the old `loss_cosine` z-scored the signals
   before computing cosine similarity, while `score_mapping`'s reported COSSIM uses the
   raw dot-product/norms (paper Eq. 15). Those are different quantities — a model could
   improve its *training* cosine loss while the *reported* COSSIM barely moved. Fixed
   by computing `loss_cosine` on the raw values, exactly like `score_mapping` does.
   (`loss_pearson` did NOT have this problem — Pearson correlation is shift/scale
   invariant by definition, so z-scoring never changed its value.)
5. **[Round 2] Generator output changed from tanh to linear**, and the iEEG target
   scaler changed from percentile-scale-and-clip to a plain per-channel z-score (same
   scheme as the CNN-BiLSTM notebooks now). The old tanh+clip combination squeezed
   every target into a hard [-1, 1] band, which specifically punishes the high-amplitude
   peaks and sharp transients that matter most for waveform similarity — the generator
   could "hide" from a hard reconstruction by staying safely inside the saturation-free
   region instead of reproducing the true signal.
6. **[Round 2] Loss weights rebalanced into a clear hierarchy** instead of ~9 similarly-
   weighted competing terms: **paired reconstruction + correlation** (`LAM2` L1,
   `LAM4`/`LAM5` pearson/cosine) now dominate; **feature-matching** (`LAM3`) is
   subordinate; **adversarial realism** (`ADV_WEIGHT`, lowered to 0.15) is weighted
   lowest, since a discriminator only judges "does this look like plausible iEEG", not
   "does this match its paired scEEG segment". `LAM1` (KL) is also lowered — strong KL
   pressure pushes the latent posterior toward a generic N(0,1) at the expense of
   precise, sample-specific reconstruction. `LAM8` (amplitude) is kept small and
   explicitly secondary: pearson/cosine already drive shape+scale matching, so giving
   amplitude too much weight just makes it compete with them for model capacity instead
   of only preventing outright amplitude collapse.
7. **[Round 2] Latent dimensionality reduced** (`LATENT_DIM`: 128→64 subject-dependent,
   256→96 pooled/LOSO). The scEEG conditioning pathway (SPADE, injected at every
   generator block) should carry most of the reconstruction-relevant information — a
   smaller `z` leaves less room for the generator to synthesize a plausible-but-
   uncorrelated waveform from the latent alone instead of actually using the
   conditioning signal.
8. Honest expectation-setting on 0.8-0.9, especially for the pooled/LOSO notebooks:
   these changes give the model the best realistic chance, but a jump from ~0.4-0.5 to
   0.8-0.9 can't be guaranteed by architecture/loss tuning alone if the underlying
   scEEG→iEEG relationship is weak for a given split — particularly LOSO, the hardest
   setting. If your earlier 0.8-0.9 numbers came from the old, leaky pooled split, they
   are not a fair target for the new leakage-free evaluation: a lower, honest number
   here is a *more* trustworthy estimate of real generalization, not evidence the model
   regressed. Use the shuffle-control diagnostic and the new per-channel breakdown
   (Section 11) to see where the real gap is, rather than assuming it's uniform.
9. **[from the previous round, unchanged]** Training schedule is just `EPOCHS` +
   `PATIENCE` (the GAN warm-up length is derived automatically inside `fit_model`,
   not a separate config knob) — set `EPOCHS` as a generous upper bound and let
   `PATIENCE` decide when a run actually stops.
10. Everything else — the VAE-cGAN architecture (paper Figs. 2-5, apart from the tanh→
   linear change above), the paper's core loss terms (Eqs. 9-13), the literal
   MSE/PCORR/COSSIM definitions (Eqs. 14-16), and the general notebook structure — is
   unchanged.


## 1. Setup & config

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os, copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

DATA_PATH = "segmented_dataset.npz"   # <-- your new dataset

# ---- per-subject split ----
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20

# ---- model size: kept modest since each subject has limited data (~200-1000 segments).
# LATENT_DIM lowered (128->64): the scEEG conditioning pathway (SPADE, at every
# generator block) should carry most of the reconstruction-relevant information, not
# the latent z -- a smaller z leaves less room for the generator to "paint" a
# plausible-but-uncorrelated waveform from the latent alone instead of actually using
# the conditioning signal. ----
ENC_Ce, ENC_IE, LATENT_DIM = 24, 3, 64
GEN_BASE_CH, GEN_IUP = 128, 2
DISC_BASE_CH, DISC_LD = 24, 4

# ---- training ----
EPOCHS, PATIENCE = 70, 18
LR = 2e-4
# lam1 (KL) lowered, lam2 (L1) raised, lam3 (feature-match) lowered, lam4/lam5
# (pearson/cosine -- cosine now aligned exactly with reported COSSIM) raised, lam6
# (mean-match, now redundant w/ fixed cosine) lowered, lam8 (amplitude) lowered to a
# light secondary role, adv_weight lowered further -- see the loss cell's markdown
# note above for the full reasoning (reconstruction+correlation > feature-match >
# adversarial realism).
LAM1, LAM2, LAM3, LAM4, LAM5, LAM6, LAM7, LAM8 = 0.01, 10.0, 2.0, 30.0, 30.0, 4.0, 3.0, 5.0
ADV_WEIGHT = 0.15
BATCH_SIZE = 32


Device: cpu


## 2. Load the segmented dataset

In [2]:
data = np.load(DATA_PATH, allow_pickle=True)
print(f"Keys found in {DATA_PATH}: {list(data.keys())}")

def _first_present(d, candidates):
    for c in candidates:
        if c in d:
            return c
    return None

# ---- flexible key lookup -----------------------------------------------
# segmented_dataset.npz is a NEW/different file from the old
# balanced_segmented_dataset.npz -- it may not use identical key names, so
# every array is looked up through a short list of common aliases instead of
# a single hard-coded key. If your actual key names aren't in these lists,
# just add them -- this is deliberately the ONLY place that needs editing.
eeg_key  = _first_present(data, ["X_eeg", "X_sc", "X_scalp", "sceeg", "scEEG"])
ieeg_key = _first_present(data, ["X_ieeg", "X_ic", "X_intracranial", "ieeg", "iEEG"])
if eeg_key is None or ieeg_key is None:
    raise KeyError(
        f"Could not find scEEG/iEEG arrays in {DATA_PATH}. Keys present: {list(data.keys())}. "
        "Add your actual key name(s) to the candidate lists above (_first_present calls)."
    )

X_eeg  = data[eeg_key].astype(np.float32)      # (N, L, M)  scEEG
X_ieeg = data[ieeg_key].astype(np.float32)     # (N, L, Mb) iEEG (target)

subj_key = _first_present(data, ["subject_ids", "subject_id", "subjects", "subj_ids"])
if subj_key is None:
    raise KeyError(f"Could not find a subject-id array in {DATA_PATH}. Keys present: {list(data.keys())}.")
subject_ids = np.asarray(data[subj_key])

# ---- labels are now OPTIONAL --------------------------------------------
# Your new segmented_dataset.npz may not carry an IED/non-IED label at all
# (unlike the old balanced_segmented_dataset.npz). This is detected instead
# of assumed: if no label array is found, HAS_LABELS=False and every
# downstream stratified split / IED-vs-Non-IED breakdown is switched off
# automatically (falls back to plain random/group splits and a
# "Combined"-only results table), instead of crashing on a KeyError.
label_key = _first_present(data, ["y", "labels", "ied_label", "ied_labels", "label"])
HAS_LABELS = label_key is not None
if HAS_LABELS:
    y = np.asarray(data[label_key]).astype(np.int64)
    print(f"Label array found (key='{label_key}') -- IED/Non-IED breakdown + stratified splits enabled.")
else:
    y = np.zeros(len(X_eeg), dtype=np.int64)   # placeholder ONLY -- never used for stratification/reporting
    print("No IED/Non-IED label array found in this dataset -- IED/Non-IED breakdown is disabled, "
          "and every split below is UNSTRATIFIED (plain random / subject-group based, not label-balanced).")

eeg_names = list(data["eeg_names"]) if "eeg_names" in data else [f"sc_ch{i}" for i in range(X_eeg.shape[2])]
fo_names  = list(data["fo_names"])  if "fo_names"  in data else [f"ie_ch{i}" for i in range(X_ieeg.shape[2])]
fs = float(data["fs"]) if "fs" in data else 256.0

L  = X_eeg.shape[1]           # time samples per segment
M  = X_eeg.shape[2]           # scEEG channels
Mb = X_ieeg.shape[2]          # iEEG channels

unique_subjects = sorted(np.unique(subject_ids).tolist())
print(f"Total segments: {len(X_eeg)}  |  scEEG shape: {X_eeg.shape}  |  iEEG shape: {X_ieeg.shape}")
print(f"Subjects ({len(unique_subjects)}):", unique_subjects)
if HAS_LABELS:
    print(f"IED: {int((y==1).sum())}   Non-IED: {int((y==0).sum())}")

# Classes reported everywhere downstream -- collapses to just "Combined"
# when this dataset has no IED/non-IED labels.
CLASSES = ["Combined", "IED", "Non-IED"] if HAS_LABELS else ["Combined"]


Keys found in segmented_dataset.npz: ['X_eeg', 'X_ieeg', 'start_idx', 'subject_ids', 'eeg_names', 'fo_names', 'fs', 'seg_len', 'gap_samples']
No IED/Non-IED label array found in this dataset -- IED/Non-IED breakdown is disabled, and every split below is UNSTRATIFIED (plain random / subject-group based, not label-balanced).
Total segments: 86400  |  scEEG shape: (86400, 64, 20)  |  iEEG shape: (86400, 64, 12)
Subjects (18): ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18']


## 3. Model architecture (VAE-cGAN, per paper Figures 2-5)

In [3]:
# ============================================================================
# VAE-cGAN architecture (paper Figures 2-5). Structure is otherwise unchanged
# from your original notebooks -- constructor args are exposed so each
# notebook can size the model to how much training data it has -- EXCEPT the
# generator's final activation (tanh -> linear, see IntracranialGenerator).
# ============================================================================

class ScalpEncoder(nn.Module):
    '''Encoder E (Fig. 2): scEEG X -> latent z via mu, sigma of a Gaussian.'''
    def __init__(self, in_ch=20, Ce=32, IE=4, latent_dim=256, L=64):
        super().__init__()
        self.conv_blocks = nn.ModuleList()
        c_in, c_out = in_ch, Ce
        for _ in range(IE):
            self.conv_blocks.append(nn.Sequential(
                nn.Conv1d(c_in, c_out, kernel_size=3, stride=2, padding=1),
                nn.InstanceNorm1d(c_out),
                nn.LeakyReLU(0.2)
            ))
            c_in, c_out = c_out, c_out * 2
        self.flatten = nn.Flatten()
        flat_size = c_in * (L // (2 ** IE))
        self.fc_mu = nn.Linear(flat_size, latent_dim)
        self.fc_lv = nn.Linear(flat_size, latent_dim)

    def reparameterize(self, mu, lv):
        std = torch.exp(0.5 * lv)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, sample=True):
        x = x.permute(0, 2, 1)                 # (B, L, M) -> (B, M, L)
        for blk in self.conv_blocks:
            x = blk(x)
        x = self.flatten(x)
        mu, lv = self.fc_mu(x), self.fc_lv(x)
        z = self.reparameterize(mu, lv) if sample else mu
        return z, mu, lv


class SPADENorm(nn.Module):
    '''SPADE (Eq. 8): activation normalized, then denormalized using
    modulation params gamma/beta inferred from the conditioning scEEG.'''
    def __init__(self, norm_ch, cond_ch, hidden_ch=64):
        super().__init__()
        self.norm = nn.InstanceNorm1d(norm_ch, affine=False)
        self.shared = nn.Sequential(nn.Conv1d(cond_ch, hidden_ch, 3, padding=1), nn.LeakyReLU(0.2))
        self.gamma = nn.Conv1d(hidden_ch, norm_ch, 3, padding=1)
        self.beta = nn.Conv1d(hidden_ch, norm_ch, 3, padding=1)

    def forward(self, A, cond):
        A_n = self.norm(A)
        feat = self.shared(cond)
        return self.gamma(feat) * A_n + self.beta(feat)


class SPADEResBlock(nn.Module):
    '''SPADE ResNet block (Fig. 3B): two SPADE blocks + Tanh + conv, residual.'''
    def __init__(self, in_ch, out_ch, cond_ch):
        super().__init__()
        mid_ch = min(in_ch, out_ch)
        self.spade0 = SPADENorm(in_ch, cond_ch)
        self.conv0 = nn.Conv1d(in_ch, mid_ch, 3, padding=1)
        self.spade1 = SPADENorm(mid_ch, cond_ch)
        self.conv1 = nn.Conv1d(mid_ch, out_ch, 3, padding=1)
        self.use_skip = in_ch != out_ch
        if self.use_skip:
            self.spade_s = SPADENorm(in_ch, cond_ch)
            self.conv_s = nn.Conv1d(in_ch, out_ch, 1, bias=False)

    def forward(self, x, cond):
        dx = self.conv0(torch.tanh(self.spade0(x, cond)))
        dx = self.conv1(torch.tanh(self.spade1(dx, cond)))
        skip = self.conv_s(torch.tanh(self.spade_s(x, cond))) if self.use_skip else x
        return dx + skip


class IntracranialGenerator(nn.Module):
    '''Generator G (Fig. 4): z -> dense -> SPADE ResNets (conditioned on
    downsampled scEEG) + upsampling -> LSTM x2 -> time-distributed dense ->
    LINEAR output (changed from tanh).

    Why linear, not tanh: the target iEEG was scaled+CLIPPED into [-1, 1] to match
    a tanh generator. That's a double restriction (percentile-scale, then clip,
    then squeeze through tanh's saturating range) that specifically punishes
    exactly the high-amplitude peaks and sharp transients that matter most for
    waveform similarity -- the generator can "hide" from a hard reconstruction by
    learning a conservative, saturation-safe waveform instead of the true one. A
    linear output removes that bottleneck; the target scaler (Section 5) is a
    plain per-channel z-score instead (no clipping), matching the CNN-BiLSTM
    notebooks' target scaling so both models are optimizing for the same thing.'''
    def __init__(self, latent_dim=256, sc_ch=20, ic_ch=12, L=64, Iup=2, base_ch=512):
        super().__init__()
        self.base_ch, self.start_len = base_ch, L // (2 ** Iup)
        self.fc = nn.Linear(latent_dim, base_ch * self.start_len)
        self.res_blocks = nn.ModuleList()
        in_ch = base_ch
        for i in range(Iup):
            out_ch = in_ch if i < Iup - 1 else in_ch // 2
            self.res_blocks.append(SPADEResBlock(in_ch, out_ch, cond_ch=sc_ch))
            in_ch = out_ch
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.lstm1 = nn.LSTM(input_size=in_ch, hidden_size=128, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=128, hidden_size=64, batch_first=True)
        self.out_fc = nn.Linear(64, ic_ch)

    def forward(self, X, z):
        B = X.shape[0]
        X_t = X.permute(0, 2, 1)               # (B, L, M) -> (B, M, L), used as SPADE condition
        h = self.fc(z).view(B, self.base_ch, self.start_len)
        cur = self.start_len
        for blk in self.res_blocks:
            cond = F.adaptive_avg_pool1d(X_t, cur)
            h = self.up(blk(h, cond))
            cur *= 2
        h, _ = self.lstm1(h.permute(0, 2, 1))   # (B, L, C) for LSTM (L time steps)
        h, _ = self.lstm2(h)
        return self.out_fc(h)                   # (B, L, Mb) synthetic iEEG, LINEAR (not tanh)


class PatchDiscriminator(nn.Module):
    '''Markovian patch discriminator (Sec. 2.3.3): classifies each 1xMb patch as real/fake.'''
    def __init__(self, ic_ch=12, ld=4, base_ch=32, n_layers=4):
        super().__init__()
        self.conv_blocks = nn.ModuleList()
        c_in, c_out = ic_ch, base_ch
        for i in range(n_layers):
            if i < n_layers - 1:
                self.conv_blocks.append(nn.Sequential(
                    nn.Conv1d(c_in, c_out, kernel_size=ld, stride=2, padding=1),
                    nn.InstanceNorm1d(c_out, affine=False),
                    nn.LeakyReLU(0.2)))
            else:
                self.conv_blocks.append(nn.Conv1d(c_in, c_out, kernel_size=ld, stride=2, padding=1))
            c_in, c_out = c_out, c_out * 2

    def forward(self, y):
        feats = []
        y = y.permute(0, 2, 1)
        for blk in self.conv_blocks:
            y = blk(y)
            feats.append(y)
        return y, feats


## 4. Loss functions (paper Eqs. 9-13, plus correlation + amplitude terms)

In [4]:
# ============================================================================
# Losses. Core terms are exactly Eqs. (9)-(13) of the paper: hinge adversarial
# loss (9)-(10), KL loss (4), L1 loss (12), feature-matching loss (13),
# combined per Eq. (11): LG = LGh + lam1*LDKL + lam2*LL1 + lam3*LFM.
#
# Beyond the paper, extra correlation/shape terms (loss_pearson, loss_cosine,
# loss_mean_match, loss_spectral, loss_amplitude) directly target the same
# metrics being reported. Set lam4=lam5=lam6=lam7=lam8=0 to fall back to the
# paper's exact Eq. (11) if you want a pure replication.
#
# REBALANCED WEIGHT HIERARCHY (per the model review): the previous defaults
# gave every term a similarly large weight, which lets 9 competing objectives
# fight each other instead of clearly prioritizing the metric you actually
# care about. The new hierarchy, reflected in the lamN defaults below AND in
# every notebook's config cell:
#
#   paired reconstruction + correlation  >  feature matching  >  adversarial realism
#      (lam2 L1, lam4 pearson, lam5 cosine)   (lam3)              (adv_weight, lowest)
#
#   with lam1 (KL) and lam8 (amplitude) both kept deliberately SMALL:
#   - KL (lam1): useful for a well-behaved latent space in general, but strong
#     KL pressure pushes the latent posterior toward a generic N(0,1), which
#     works against precise, sample-specific conditional reconstruction --
#     lowered so it regularizes without dominating.
#   - amplitude (lam8): loss_pearson/(the now-fixed) loss_cosine already push
#     hard on shape+scale; amplitude is a light secondary constraint against
#     amplitude collapse, not a primary objective -- if it's weighted too high
#     it competes with shape-matching for model capacity instead of just
#     preventing the flat-output failure mode.
#
# loss_cosine below is also now aligned EXACTLY with the reported COSSIM (see
# its docstring) -- this was flagged as the single biggest source of
# train/eval mismatch: the old z-scored cosine loss optimized a different
# quantity than the dot-product/norm COSSIM actually being reported.
# ============================================================================

def loss_disc(out_real, out_fake):
    return (-torch.mean(torch.clamp(-1 + out_real, max=0))
            - torch.mean(torch.clamp(-1 - out_fake, max=0)))

def loss_gen_hinge(out_fake):
    return -torch.mean(out_fake)

def loss_kl(mu, lv):
    return -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())

def loss_l1(y_real, y_est):
    return torch.mean(torch.abs(y_real - y_est))

def loss_feat_match(feats_real, feats_fake):
    lfm = sum(torch.mean(torch.abs(fr - ff)) for fr, ff in zip(feats_real, feats_fake))
    return lfm / len(feats_real)

def zscore_time(x, eps=1e-6):
    mu = x.mean(dim=1, keepdim=True)
    sd = x.std(dim=1, keepdim=True)
    return (x - mu) / (sd + eps)

def loss_pearson(y_real, y_est, eps=1e-8):
    '''Pearson correlation coefficient is itself shift/scale invariant (it mean-centers
    and normalizes internally), so this is exactly PCORR (paper Eq. 16) either way --
    no z-scoring/alignment issue here, unlike loss_cosine below.'''
    yr = y_real - y_real.mean(dim=1, keepdim=True)
    ye = y_est - y_est.mean(dim=1, keepdim=True)
    num = (yr * ye).sum(dim=1)
    den = torch.sqrt((yr ** 2).sum(dim=1) * (ye ** 2).sum(dim=1) + eps)
    return 1 - (num / (den + eps)).mean()

def loss_cosine(y_real, y_est, eps=1e-8):
    '''Matches the EXACT COSSIM definition used in score_mapping/evaluation (paper
    Eq. 15): raw dot product over raw norms -- NOT cosine similarity of z-scored
    signals. The earlier version of this loss z-scored y_real/y_est first, which
    discards the DC offset/mean before computing cosine similarity -- so the network
    was being optimized against a DIFFERENT quantity than the one actually reported
    ("make the z-scored waveforms similar" vs. "make COSSIM high"). This was the
    single biggest source of train/eval mismatch (a model could improve its training
    cosine loss while reported COSSIM stayed flat). This version is exactly what's
    reported, so improving this loss now directly improves the reported metric.'''
    num = (y_real * y_est).sum(dim=1)
    den = torch.norm(y_real, dim=1) * torch.norm(y_est, dim=1)
    return 1 - (num / (den + eps)).mean()

def loss_mean_match(y_real, y_est):
    '''Kept, but now largely redundant with the fixed (non-z-scored) loss_cosine
    above, since raw-value cosine similarity already implicitly penalizes a DC-offset
    mismatch. Weighted low (see lam6 default below / config cells) to avoid two loss
    terms fighting over the same thing.'''
    return torch.mean((y_real.mean(dim=1) - y_est.mean(dim=1)) ** 2)

def loss_spectral(y_real, y_est):
    '''L1 distance between rfft magnitude spectra. Complementary to the
    time-domain L1/Pearson/cosine terms -- encourages matching frequency content.
    Tier-2 / secondary term, weighted low (see lam7 default below).'''
    Yr = torch.fft.rfft(y_real, dim=1).abs()
    Ye = torch.fft.rfft(y_est, dim=1).abs()
    return torch.mean(torch.abs(Yr - Ye))

def loss_amplitude(y_real, y_est, eps=1e-8):
    '''Relative per-segment, per-channel amplitude-matching loss. Kept as a light
    SECONDARY constraint (small lam8, see below/config cells) against the model
    collapsing its output amplitude toward zero -- NOT weighted as heavily as
    pearson/cosine, since those already drive shape+scale matching and giving
    amplitude too much weight makes it compete with shape-matching for capacity
    instead of just preventing the flat-output failure mode.'''
    real_amp = y_real.std(dim=1) + eps         # (B, Mb)
    est_amp  = y_est.std(dim=1) + eps          # (B, Mb)
    return torch.mean(((real_amp - est_amp) / real_amp) ** 2)

def loss_gen_total(out_fake, mu, lv, y_real, y_est, feats_real, feats_fake,
                    lam1=0.01, lam2=10.0, lam3=2.0, lam4=30.0, lam5=30.0, lam6=4.0, lam7=3.0,
                    lam8=5.0, adv_weight=0.15):
    '''Defaults reflect the rebalanced hierarchy described above: lam2/lam4/lam5
    (reconstruction + correlation) dominate; lam3 (feature matching) and adv_weight
    (pure adversarial realism) are intentionally low, in that order, so the GAN
    cannot trade fidelity (what PCORR/COSSIM/MSE actually measure) for "realism";
    lam1 (KL) and lam8 (amplitude) are both small secondary/regularizing terms.
    Every notebook's config cell sets these explicitly -- these function defaults
    are just a sane fallback if fit_model is ever called without them.'''
    lgh  = loss_gen_hinge(out_fake) * adv_weight
    ldkl = loss_kl(mu, lv)
    ll1  = loss_l1(y_real, y_est)
    lfm  = loss_feat_match(feats_real, feats_fake)
    lpc  = loss_pearson(y_real, y_est)
    lcos = loss_cosine(y_real, y_est)
    lmn  = loss_mean_match(y_real, y_est)
    lspec = loss_spectral(y_real, y_est)
    lamp = loss_amplitude(y_real, y_est)
    total = (lgh + lam1*ldkl + lam2*ll1 + lam3*lfm + lam4*lpc + lam5*lcos
             + lam6*lmn + lam7*lspec + lam8*lamp)
    return total, lgh, ldkl, ll1, lfm, lpc, lcos, lmn, lspec, lamp


In [5]:
class SegSet(Dataset):
    def __init__(self, sc, ie, lab):
        self.sc  = torch.tensor(sc,  dtype=torch.float32)
        self.ie  = torch.tensor(ie,  dtype=torch.float32)
        self.lab = torch.tensor(lab, dtype=torch.float32)
    def __len__(self): return len(self.lab)
    def __getitem__(self, i): return self.sc[i], self.ie[i], self.lab[i]


## 5. Per-channel scaling

In [6]:
# ============================================================================
# Per-channel normalization.
#
#   - scEEG (model INPUT): per-channel z-score (mean/std from TRAIN only) -- unchanged.
#   - iEEG (model TARGET): per-channel z-score (mean/std from TRAIN only) -- CHANGED
#     from percentile-scale + clip-to-[-1,1]. That scheme existed only to match a
#     tanh-bounded generator output; now that the generator's output is linear (see
#     the architecture cell), there's no need to clip the target into a bounded
#     range, and NOT clipping means the model is never asked to reproduce a
#     distorted/clamped version of high-amplitude peaks and sharp transients --
#     exactly the morphology that matters most for waveform similarity. This also
#     makes the VAE-cGAN and CNN-BiLSTM notebooks target the SAME representation.
# ============================================================================

def fit_scaler_sc(X_train):
    '''scEEG: returns (mean, std), computed per-channel over TRAIN only.'''
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std  = X_train.std(axis=(0, 1), keepdims=True) + 1e-10
    return mean, std

def apply_scaler_sc(X, mean, std):
    return (X - mean) / std

def fit_scaler_ie(X_train):
    '''iEEG: returns (mean, std), computed per-channel over TRAIN only. Same
    z-score scheme as fit_scaler_sc -- no percentile scaling, no clipping.'''
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std  = X_train.std(axis=(0, 1), keepdims=True) + 1e-10
    return mean, std

def apply_scaler_ie(X, mean, std):
    return (X - mean) / std


## 6. Metrics: MSE / PCORR / COSSIM

In [7]:
def score_mapping(enc, gen, loader, device=DEVICE):
    '''Returns MSE, PCORR, COSSIM for "Combined", and ALSO for "IED"/"Non-IED" when
    this dataset actually has labels (HAS_LABELS, set in the data-loading cell) --
    these are the LITERAL definitions from the paper (Eqs. 14-16), computed directly
    on the values in the same consistent scale the network trains on (Section 5
    scalers, fit once on the training set), NOT re-standardized per segment:
      - MSE  = mean((y - y_est)^2)                        [paper Eq. 14]
      - COSSIM = dot(y, y_est) / (||y|| * ||y_est||)       [paper Eq. 15]
      - PCORR  = pearson correlation(y, y_est)             [paper Eq. 16]
    '''
    enc.eval(); gen.eval()
    mse_vals, pcorr_vals, cos_vals, label_vals = [], [], [], []
    with torch.no_grad():
        for sc, ie, lab in loader:
            sc, ie = sc.to(device), ie.to(device)
            z, mu, lv = enc(sc, sample=False)
            y_est = gen(sc, z)
            ie_np, ye_np = ie.cpu().numpy(), y_est.cpu().numpy()
            for i in range(ie_np.shape[0]):
                for j in range(ie_np.shape[2]):
                    yv, yev = ie_np[i, :, j], ye_np[i, :, j]
                    mse_vals.append(np.mean((yv - yev) ** 2))
                    pcorr_vals.append(pearsonr(yv, yev)[0])
                    denom = np.linalg.norm(yv) * np.linalg.norm(yev)
                    cos_vals.append(np.dot(yv, yev) / (denom + 1e-8))
                    label_vals.append(lab[i].item())
    mse_vals, pcorr_vals, cos_vals, label_vals = (np.array(a) for a in
        (mse_vals, pcorr_vals, cos_vals, label_vals))

    def summarize(mask):
        if mask.sum() == 0:
            return dict(MSE=np.nan, PCORR=np.nan, COSSIM=np.nan)
        return dict(MSE=float(np.mean(mse_vals[mask])),
                    PCORR=float(np.mean(pcorr_vals[mask])),
                    COSSIM=float(np.mean(cos_vals[mask])))

    out = {"Combined": summarize(np.ones_like(label_vals, dtype=bool))}
    if HAS_LABELS:
        out["IED"] = summarize(label_vals == 1)
        out["Non-IED"] = summarize(label_vals == 0)
    return out


## 7. Training loop

In [8]:
def fit_model(enc, gen, disc, train_loader, val_loader, device=DEVICE,
              epochs=100, lr=2e-4, lam1=0.05, lam2=6.0, lam3=3.0, lam4=35.0, lam5=40.0,
              lam6=10.0, lam7=5.0, lam8=20.0, adv_weight=0.3, patience=20,
              grad_clip=5.0, verbose=True):
    '''Standard VAE-cGAN training loop (hinge GAN with warm-up before the
    adversarial term switches on). ONLY TWO SCHEDULE KNOBS ARE EXPOSED --
    `epochs` (a generous upper bound / safety cap) and `patience` (how many
    epochs to wait for a validation improvement before stopping) -- which is
    standard early-stopping practice: set epochs high enough that patience is
    what actually decides when training ends, rather than hand-tuning several
    separate schedule numbers. The GAN warm-up length (how many initial
    epochs run with the adversarial term off, for stability) is no longer a
    separate knob -- it's derived automatically as a modest fraction of
    `epochs` below, since it's an implementation detail of hinge-GAN training
    rather than something that needs manual tuning per run.
    Checkpoint selection tracks validation PCORR + COSSIM directly, since
    that's what's ultimately reported.'''
    warmup_epochs = max(5, int(0.2 * epochs))   # auto-derived, not a separate config knob
    enc, gen, disc = enc.to(device), gen.to(device), disc.to(device)
    opt_eg = torch.optim.Adam(list(enc.parameters()) + list(gen.parameters()), lr=lr, betas=(0.5, 0.999))
    opt_disc = torch.optim.Adam(disc.parameters(), lr=lr * 0.5, betas=(0.5, 0.999))
    sched_eg = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_eg, mode='min', factor=0.5, patience=8)

    best_score, wait = -float('inf'), 0
    best_enc_state, best_gen_state = None, None

    for epoch in range(epochs):
        enc.train(); gen.train(); disc.train()
        use_adv = epoch >= warmup_epochs
        tr_d = tr_g = 0.0

        for sc, ie, _ in train_loader:
            sc, ie = sc.to(device), ie.to(device)
            z, mu, lv = enc(sc)
            y_est = gen(sc, z)

            if use_adv:
                o_r, f_r = disc(ie)
                o_fd, f_fd = disc(y_est.detach())
                ld = loss_disc(o_r, o_fd)
                opt_disc.zero_grad(); ld.backward()
                torch.nn.utils.clip_grad_norm_(disc.parameters(), grad_clip)
                opt_disc.step()
            else:
                ld = torch.tensor(0.0, device=device)

            o_f, f_f = disc(y_est)
            o_r2, f_r2 = disc(ie)
            if not use_adv:
                o_f = o_f.detach() * 0.0

            lg, *_ = loss_gen_total(o_f, mu, lv, ie, y_est, f_r2, f_f,
                                     lam1, lam2, lam3, lam4, lam5, lam6, lam7, lam8, adv_weight)
            opt_eg.zero_grad(); lg.backward()
            torch.nn.utils.clip_grad_norm_(list(enc.parameters()) + list(gen.parameters()), grad_clip)
            opt_eg.step()

            tr_d += ld.item(); tr_g += lg.item()

        tr_d /= len(train_loader); tr_g /= len(train_loader)

        enc.eval(); gen.eval(); disc.eval()
        va_g = va_corr = va_cos = 0.0
        with torch.no_grad():
            for sc, ie, _ in val_loader:
                sc, ie = sc.to(device), ie.to(device)
                z, mu, lv = enc(sc)
                y_est = gen(sc, z)
                o_f, f_f = disc(y_est)
                o_r, f_r = disc(ie)
                lg, *_ = loss_gen_total(o_f, mu, lv, ie, y_est, f_r, f_f,
                                         lam1, lam2, lam3, lam4, lam5, lam6, lam7, lam8, adv_weight)
                va_g += lg.item()
                va_corr += (1 - loss_pearson(ie, y_est)).item()
                va_cos += (1 - loss_cosine(ie, y_est)).item()
        va_g /= len(val_loader); va_corr /= len(val_loader); va_cos /= len(val_loader)
        sched_eg.step(va_g)
        val_score = (va_corr + va_cos) / 2

        if val_score > best_score:
            best_score, wait = val_score, 0
            best_enc_state = copy.deepcopy(enc.state_dict())
            best_gen_state = copy.deepcopy(gen.state_dict())
        else:
            wait += 1

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:3d} | train_D {tr_d:.3f} train_G {tr_g:.3f} "
                  f"| val_PCORR {va_corr:.3f} val_COSSIM {va_cos:.3f}")

        if wait >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

    if best_enc_state is not None:
        enc.load_state_dict(best_enc_state)
        gen.load_state_dict(best_gen_state)
    return enc, gen, disc


In [9]:
def metrics_dict_to_row(d):
    '''Flatten {"Combined":{...}, ["IED":{...}, "Non-IED":{...}]} into one flat row.
    Only includes IED/Non-IED columns when this dataset actually has labels
    (CLASSES is set in the data-loading cell based on HAS_LABELS).'''
    row = {}
    for cls in CLASSES:
        for met in ["MSE", "PCORR", "COSSIM"]:
            row[(met, cls)] = d[cls][met]
    return row

def build_results_table(rows_dict, index_name="Subject"):
    '''rows_dict: {row_label: metrics_dict}. Returns a DataFrame with a
    (metric, class) MultiIndex column layout, plus a trailing Mean row.'''
    flat_rows = {label: metrics_dict_to_row(d) for label, d in rows_dict.items()}
    df = pd.DataFrame.from_dict(flat_rows, orient="index")
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["Metric", "Class"])
    df.index.name = index_name
    mean_row = df.mean(numeric_only=True)
    df.loc["Mean"] = mean_row
    return df.round(3)


## 8. Per-subject 70/10/20 split

For each subject: a single `train_test_split` (stratified by IED label if available)
carves out 20% test, then a further split of the remaining 80% carves out 10%
validation / 70% training. scEEG/iEEG are scaled using each subject's own training
data only (Section 5) — never validation/test statistics.


In [10]:
from sklearn.model_selection import train_test_split

def make_subject_split(sc, ie, lab, test_size=TEST_FRAC, val_size=VAL_FRAC, seed=SEED):
    strat1 = lab if HAS_LABELS else None
    Xtr_sc, Xte_sc, Xtr_ie, Xte_ie, ytr, yte = train_test_split(
        sc, ie, lab, test_size=test_size, random_state=seed, stratify=strat1)
    val_ratio = val_size / (1 - test_size)
    strat2 = ytr if HAS_LABELS else None
    Xtr_sc, Xva_sc, Xtr_ie, Xva_ie, ytr, yva = train_test_split(
        Xtr_sc, Xtr_ie, ytr, test_size=val_ratio, random_state=seed, stratify=strat2)
    return (Xtr_sc, Xtr_ie, ytr, Xva_sc, Xva_ie, yva, Xte_sc, Xte_ie, yte)

subject_splits = {}
for subj in unique_subjects:
    mask = subject_ids == subj
    subject_splits[subj] = make_subject_split(X_eeg[mask], X_ieeg[mask], y[mask])
    sp = subject_splits[subj]
    print(f"{subj}: train={len(sp[2])} val={len(sp[5])} test={len(sp[8])}")


sub-01: train=3360 val=480 test=960
sub-02: train=3360 val=480 test=960
sub-03: train=3360 val=480 test=960
sub-04: train=3360 val=480 test=960
sub-05: train=3360 val=480 test=960
sub-06: train=3360 val=480 test=960
sub-07: train=3360 val=480 test=960
sub-08: train=3360 val=480 test=960
sub-09: train=3360 val=480 test=960
sub-10: train=3360 val=480 test=960
sub-11: train=3360 val=480 test=960
sub-12: train=3360 val=480 test=960
sub-13: train=3360 val=480 test=960
sub-14: train=3360 val=480 test=960
sub-15: train=3360 val=480 test=960
sub-16: train=3360 val=480 test=960
sub-17: train=3360 val=480 test=960
sub-18: train=3360 val=480 test=960


## 9. Train one VAE-cGAN per subject

In [ ]:
per_subject_metrics = {}
trained_models = {}

for subj in unique_subjects:
    print(f"\n=== {subj} ===")
    Xtr_sc, Xtr_ie, ytr, Xva_sc, Xva_ie, yva, Xte_sc, Xte_ie, yte = subject_splits[subj]

    sc_mean, sc_std = fit_scaler_sc(Xtr_sc)
    ie_mean, ie_std = fit_scaler_ie(Xtr_ie)
    Xtr_sc_n, Xva_sc_n, Xte_sc_n = (apply_scaler_sc(x, sc_mean, sc_std) for x in (Xtr_sc, Xva_sc, Xte_sc))
    Xtr_ie_n, Xva_ie_n, Xte_ie_n = (apply_scaler_ie(x, ie_mean, ie_std) for x in (Xtr_ie, Xva_ie, Xte_ie))

    tr_loader = DataLoader(SegSet(Xtr_sc_n, Xtr_ie_n, ytr), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(SegSet(Xva_sc_n, Xva_ie_n, yva), batch_size=16, shuffle=False)
    te_loader = DataLoader(SegSet(Xte_sc_n, Xte_ie_n, yte), batch_size=16, shuffle=False)

    enc = ScalpEncoder(in_ch=M, Ce=ENC_Ce, IE=ENC_IE, latent_dim=LATENT_DIM, L=L)
    gen = IntracranialGenerator(latent_dim=LATENT_DIM, sc_ch=M, ic_ch=Mb, L=L, Iup=GEN_IUP, base_ch=GEN_BASE_CH)
    disc = PatchDiscriminator(ic_ch=Mb, ld=DISC_LD, base_ch=DISC_BASE_CH)

    enc, gen, disc = fit_model(enc, gen, disc, tr_loader, va_loader, device=DEVICE,
                                epochs=EPOCHS, lr=LR, lam1=LAM1, lam2=LAM2, lam3=LAM3,
                                lam4=LAM4, lam5=LAM5, lam6=LAM6, lam7=LAM7, lam8=LAM8,
                                adv_weight=ADV_WEIGHT, patience=PATIENCE, verbose=False)

    metrics = score_mapping(enc, gen, te_loader, device=DEVICE)
    per_subject_metrics[subj] = metrics
    trained_models[subj] = (enc, gen, disc, te_loader)
    print(f"  test -> Combined: MSE={metrics['Combined']['MSE']:.3f} "
          f"PCORR={metrics['Combined']['PCORR']:.3f} COSSIM={metrics['Combined']['COSSIM']:.3f}")



=== sub-01 ===


## 10. Results table — MSE / PCORR / COSSIM

In [ ]:
results_table = build_results_table(per_subject_metrics, index_name="Subject")
display(results_table)


### Per-channel breakdown (example subject)

A single poor channel can pull the Combined average down a lot even when most
channels are already matching well (e.g. three channels at ~0.85 and one at ~0.20
average to only ~0.66). This shows PCORR/COSSIM/MSE broken out by iEEG channel for
one subject, sorted worst-to-best, so a uniformly "bad" Combined score can be told
apart from a few problem channels dragging down an otherwise good result.


In [ ]:
def channel_breakdown(enc, gen, loader, device=DEVICE):
    '''Per-CHANNEL average MSE/PCORR/COSSIM (as opposed to score_mapping's overall
    average across every segment AND channel). Useful diagnostic per the model review:
    a single very poor channel can pull a Combined average down substantially even when
    most channels are already matching well -- e.g. three channels at ~0.85 and one at
    ~0.20 average to only ~0.66. This shows whether that's happening, and if so, which
    channel(s) are the actual problem, rather than treating the model as uniformly bad.'''
    enc.eval(); gen.eval()
    per_ch = {j: {"mse": [], "pcorr": [], "cos": []} for j in range(Mb)}
    with torch.no_grad():
        for sc, ie, _ in loader:
            sc, ie = sc.to(device), ie.to(device)
            z, mu, lv = enc(sc, sample=False)
            y_est = gen(sc, z)
            ie_np, ye_np = ie.cpu().numpy(), y_est.cpu().numpy()
            for j in range(ie_np.shape[2]):
                for i in range(ie_np.shape[0]):
                    yv, yev = ie_np[i, :, j], ye_np[i, :, j]
                    per_ch[j]["mse"].append(np.mean((yv - yev) ** 2))
                    per_ch[j]["pcorr"].append(pearsonr(yv, yev)[0])
                    denom = np.linalg.norm(yv) * np.linalg.norm(yev)
                    per_ch[j]["cos"].append(np.dot(yv, yev) / (denom + 1e-8))
    rows = {}
    for j in range(Mb):
        name = fo_names[j] if j < len(fo_names) else f"ch{j}"
        rows[name] = {
            "MSE": float(np.mean(per_ch[j]["mse"])),
            "PCORR": float(np.mean(per_ch[j]["pcorr"])),
            "COSSIM": float(np.mean(per_ch[j]["cos"])),
        }
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index.name = "iEEG channel"
    return df.round(3).sort_values("PCORR")

example_subj_cb = unique_subjects[0]
enc_cb, gen_cb, _, te_loader_cb = trained_models[example_subj_cb]
display(channel_breakdown(enc_cb, gen_cb, te_loader_cb, device=DEVICE))


### Shuffle-control diagnostic (run this on YOUR real data)

The single most direct way to answer "is there real learnable signal in my
scEEG<->iEEG pairing, or is something wrong?" Trains a second model, identical in
every way, except the iEEG targets are **randomly shuffled** so each scEEG segment is
paired with the WRONG iEEG segment on purpose, then compares its test PCORR against
the real model's test PCORR above.

- **Shuffled PCORR << Real PCORR:** the model is learning genuine shared structure.
- **Shuffled PCORR ≈ Real PCORR:** very little real signal is being used — check the
  data preparation (verify `X_eeg[k]`/`X_ieeg[k]` really are the same segment/subject).


In [ ]:
example_subj = unique_subjects[0]
Xtr_sc, Xtr_ie, ytr, Xva_sc, Xva_ie, yva, Xte_sc, Xte_ie, yte = subject_splits[example_subj]

sc_mean, sc_std = fit_scaler_sc(Xtr_sc)
ie_mean, ie_std = fit_scaler_ie(Xtr_ie)
Xtr_sc_n, Xva_sc_n, Xte_sc_n = (apply_scaler_sc(x, sc_mean, sc_std) for x in (Xtr_sc, Xva_sc, Xte_sc))
Xtr_ie_n, Xva_ie_n, Xte_ie_n = (apply_scaler_ie(x, ie_mean, ie_std) for x in (Xtr_ie, Xva_ie, Xte_ie))

val_loader = DataLoader(SegSet(Xva_sc_n, Xva_ie_n, yva), batch_size=16, shuffle=False)
test_loader = DataLoader(SegSet(Xte_sc_n, Xte_ie_n, yte), batch_size=16, shuffle=False)
test_metrics = per_subject_metrics[example_subj]

print(f"Running shuffle-control diagnostic on {example_subj}...")

rng_shuffle = np.random.RandomState(SEED)
shuffle_idx = rng_shuffle.permutation(len(Xtr_ie_n))
Xtr_ie_shuffled = Xtr_ie_n[shuffle_idx]   # scrambles which iEEG target goes with which scEEG input

train_loader_shuf = DataLoader(SegSet(Xtr_sc_n, Xtr_ie_shuffled, ytr), batch_size=BATCH_SIZE, shuffle=True)

enc_shuf = ScalpEncoder(in_ch=M, Ce=ENC_Ce, IE=ENC_IE, latent_dim=LATENT_DIM, L=L)
gen_shuf = IntracranialGenerator(latent_dim=LATENT_DIM, sc_ch=M, ic_ch=Mb, L=L, Iup=GEN_IUP, base_ch=GEN_BASE_CH)
disc_shuf = PatchDiscriminator(ic_ch=Mb, ld=DISC_LD, base_ch=DISC_BASE_CH)

enc_shuf, gen_shuf, _ = fit_model(enc_shuf, gen_shuf, disc_shuf, train_loader_shuf, val_loader, device=DEVICE,
                                   epochs=max(20, EPOCHS // 2), lr=LR, lam1=LAM1, lam2=LAM2, lam3=LAM3,
                                   lam4=LAM4, lam5=LAM5, lam6=LAM6, lam7=LAM7, lam8=LAM8, adv_weight=ADV_WEIGHT,
                                   patience=max(10, PATIENCE // 2), verbose=False)

shuffled_metrics = score_mapping(enc_shuf, gen_shuf, test_loader, device=DEVICE)
print("Shuffled-pairing control -> Combined: "
      f"MSE={shuffled_metrics['Combined']['MSE']:.3f}  PCORR={shuffled_metrics['Combined']['PCORR']:.3f}  "
      f"COSSIM={shuffled_metrics['Combined']['COSSIM']:.3f}")
print("Real-pairing model (from above) -> Combined: "
      f"MSE={test_metrics['Combined']['MSE']:.3f}  PCORR={test_metrics['Combined']['PCORR']:.3f}  "
      f"COSSIM={test_metrics['Combined']['COSSIM']:.3f}")
if test_metrics['Combined']['PCORR'] - shuffled_metrics['Combined']['PCORR'] > 0.15:
    print("\n-> Real pairing clearly beats shuffled pairing: the model is using genuine signal.")
else:
    print("\n-> Real pairing is NOT clearly better than shuffled pairing -- investigate the "
          "data preparation (see markdown above) before tuning the model further.")


## 12. Save results and models

In [ ]:
results_table.to_csv(os.path.join(".", "vae_cgan_nbd_subject_dependent_results.csv"))
torch.save({subj: {"enc": m[0].state_dict(), "gen": m[1].state_dict()}
            for subj, m in trained_models.items()},
           os.path.join(".", "vae_cgan_nbd_subject_dependent_models.pt"))
print("Saved: vae_cgan_nbd_subject_dependent_results.csv, vae_cgan_nbd_subject_dependent_models.pt")


## 13. Sanity check: real vs. estimated iEEG for one subject

In [ ]:
def plot_real_vs_est(enc, gen, loader, ch=0, n=3, title=""):
    enc.eval(); gen.eval()
    sc, ie, lab = next(iter(loader))
    with torch.no_grad():
        z, mu, lv = enc(sc.to(DEVICE), sample=False)
        y_est = gen(sc.to(DEVICE), z).cpu().numpy()
    ie_np = ie.numpy()
    fig, axes = plt.subplots(1, n, figsize=(4*n, 3))
    for i in range(n):
        axes[i].plot(ie_np[i, :, ch], label="Real iEEG", color="black")
        axes[i].plot(y_est[i, :, ch], label="Estimated iEEG", color="crimson", alpha=0.8)
        lab_txt = f"label={int(lab[i].item())}" if HAS_LABELS else f"segment #{i}"
        axes[i].set_title(lab_txt)
        axes[i].legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
example_subj = unique_subjects[0]
enc_ex, gen_ex, _, te_loader_ex = trained_models[example_subj]
plot_real_vs_est(enc_ex, gen_ex, te_loader_ex, ch=0, title=f"{example_subj} - test set, iEEG channel {fo_names[0]}")
